# Experimento: Clave de deduplicación de MATRICULADOS

**Objetivo:** validar si forzar `PERIODO_ESTANDARIZADO` como parte de la clave candidata elimina los duplicados residuales de matriculados.

**Contexto:** el contrato `matriculados_schema.json` autodetectó una clave *sin período* que deja **75.907 duplicados** (sobre los 12 archivos). Este experimento la pone a prueba con 2 semestres distintos (2025-I y 2025-II).

> **Importante:** `PERIODO_ESTANDARIZADO` es *constante dentro de cada archivo* (cada CSV es un semestre). Por eso se necesitan **al menos 2 semestres** para aislar el efecto del período en la clave. Con 1 solo archivo, ambas claves darían el mismo resultado y el test no sería concluyente.

In [1]:
from pathlib import Path

# Detección automática de la raíz del proyecto
current_dir = Path.cwd()
if (current_dir / "data").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta 'data'. Ejecuta este notebook desde la raíz "
        "del proyecto o desde notebooks/."
    )

SCHEMAS = PROJECT_ROOT / "data" / "schemas"
MAT_DIR = PROJECT_ROOT / "data" / "Bronce" / "matriculados_raw"
SCHEMA_PATH = SCHEMAS / "matriculados_schema.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAT_DIR:", MAT_DIR)
print("SCHEMAS existe:", SCHEMAS.exists())
print("MAT_DIR existe:", MAT_DIR.exists())
print("SCHEMA_PATH existe:", SCHEMA_PATH.exists())

PROJECT_ROOT: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP
MAT_DIR: /mnt/datos/Proyectos/A.Prueba Tecnica UCSP/data/Bronce/matriculados_raw
SCHEMAS existe: True
MAT_DIR existe: True
SCHEMA_PATH existe: True


## Hipótesis

- **Clave VIEJA** (autodetectada, sin período): `CODIGO_INEI + CODIGO_SIU_PROGRAMA + CODIGO_GRUPO_1 + CODIGO_GRUPO_3 + CODIGO_LOCAL + GUID_PERSONA` → deja 75.907 duplicados (12 archivos).
- **Clave NUEVA** (forzando período): `[PERIODO_ESTANDARIZADO] + clave_vieja` → se esperan **0 duplicados**.

Si la clave NUEVA da 0, confirmamos que la granularidad real de matriculados es *persona + programa + local + período* (cada fila = matrícula de una persona en un programa en un semestre).

In [2]:
import json

with SCHEMA_PATH.open("r", encoding="utf-8") as fh:
    schema = json.load(fh)

clave_vieja = schema["granularidad"]["clave_candidata"]
columna_periodo = schema["columna_periodo"]
clave_nueva = [columna_periodo] + clave_vieja

print("columna_periodo:", columna_periodo)
print("clave VIEJA (%d cols):" % len(clave_vieja), clave_vieja)
print("clave NUEVA (%d cols):" % len(clave_nueva), clave_nueva)
print("duplicados_con_clave (JSON, 12 archivos):", f"{schema['granularidad']['duplicados_con_clave']:,}")
print("duplicados_exactos_total (JSON):", f"{schema['calidad']['duplicados_exactos_total']:,}")
print("separador:", repr(schema["separador"]), "| encoding:", schema["encoding"])

columna_periodo: PERIODO_ESTANDARIZADO
clave VIEJA (6 cols): ['CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']
clave NUEVA (7 cols): ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']
duplicados_con_clave (JSON, 12 archivos): 75,907
duplicados_exactos_total (JSON): 902,447
separador: '|' | encoding: latin-1


In [3]:
archivos = sorted(MAT_DIR.glob("*.csv"))
if len(archivos) < 2:
    raise ValueError(
        f"Se necesitan al menos 2 archivos semestrales para el experimento (hay {len(archivos)})."
    )

# Los 2 semestres más recientes: 2025-I y 2025-II
archivos_prueba = archivos[-2:]
print("Archivos de prueba (2 últimos semestres):")
for f in archivos_prueba:
    print("  -", f.name)

Archivos de prueba (2 últimos semestres):
  - matriculado_2025_I.csv
  - matriculado_2025_II.csv


In [4]:
import pandas as pd

lista = [
    pd.read_csv(f, sep=schema["separador"], encoding=schema["encoding"], low_memory=False)
    for f in archivos_prueba
]
df = pd.concat(lista, ignore_index=True)
del lista

print("Shape (2 semestres):", df.shape)
print("\nDistribución de", columna_periodo + ":")
print(df[columna_periodo].value_counts().to_string())

# Sanity check: el período debe variar para que el test sea válido
if df[columna_periodo].nunique() < 2:
    raise ValueError("PERIODO_ESTANDARIZADO es constante: el test no es válido.")
print("\nOK: PERIODO_ESTANDARIZADO tiene", df[columna_periodo].nunique(),
      "valores distintos → test válido.")

Shape (2 semestres): (3445312, 39)

Distribución de PERIODO_ESTANDARIZADO:
PERIODO_ESTANDARIZADO
2025-1    1752734
2025-2    1692578

OK: PERIODO_ESTANDARIZADO tiene 2 valores distintos → test válido.


## Limpieza básica (simula el ETL)

No es necesaria para probar la clave, pero reproduce el entorno real del pipeline
(`strip` + `MAYÚSCULAS` + sin tildes) para que los conteos sean comparables con el
script de producción. Se aplica solo a las columnas string detectadas dinámicamente.

In [ ]:
import unicodedata


def normalizar_serie(s):
    # dtype object: usa el motor regex de Python (soporta \u). El dtype 'str' de
    # pandas 3 usa backend Arrow/pyarrow, cuya regex no soporta esos escapes.
    return (
        s.astype(object)
        .str.strip()
        .str.upper()
        .str.normalize("NFD")
        .str.replace(r"[\u0300-\u036f]", "", regex=True)
    )


cols_str = list(df.select_dtypes(include=["object", "string"]).columns)
print(f"Columnas string a normalizar: {len(cols_str)}")
for col in cols_str:
    df[col] = normalizar_serie(df[col])
print("Normalización completada.")

Columnas string a normalizar: 26


## Mediciones

Se comparan **4 conteos** sobre el mismo DataFrame (2 semestres):

1. **Duplicados exactos** (todas las columnas): contexto de suciedad del dato.
2. **Duplicados con clave VIEJA** (sin período): debería ser alto (colisiones entre semestres + NaN).
3. **Duplicados con clave NUEVA** (con período): debe coincidir con los exactos.
4. **Residual con clave NUEVA tras eliminar exactos**: esperamos **0** → confirma que no quedan duplicados semánticos.

> Nota técnica: `drop_duplicates` de pandas trata `NaN == NaN`. La clave VIEJA incluye
> `CODIGO_GRUPO_1` y `CODIGO_GRUPO_3` (con ~3.5M de nulos en el total), lo que **infla**
> su conteo. Por eso el test es **comparativo**: la clave NUEVA debe reducir el conteo
> hasta el nivel exacto de los duplicados exactos (nada más, nada menos).

In [ ]:
n = len(df)
dup_exactos = n - len(df.drop_duplicates())
print(f"Duplicados EXACTOS (todas las columnas): {dup_exactos:,} ({dup_exactos / n:.2%})")

Duplicados EXACTOS (todas las columnas): 149,539 (4.34%)


In [ ]:
dup_clave_vieja = n - len(df.drop_duplicates(subset=clave_vieja))
print(f"Duplicados clave VIEJA (sin periodo): {dup_clave_vieja:,} ({dup_clave_vieja / n:.2%})")
print("Clave VIEJA:", clave_vieja)

Duplicados clave VIEJA (sin periodo): 1,493,735 (43.36%)
Clave VIEJA: ['CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']


In [ ]:
dup_clave_nueva = n - len(df.drop_duplicates(subset=clave_nueva))
print(f"Duplicados clave NUEVA (con periodo): {dup_clave_nueva:,} ({dup_clave_nueva / n:.2%})")
print("Clave NUEVA:", clave_nueva)

Duplicados clave NUEVA (con periodo): 149,539 (4.34%)
Clave NUEVA: ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']


In [ ]:
# Residual de la clave NUEVA sobre datos SIN duplicados exactos
df_sin_exactos = df.drop_duplicates()
dup_residual_nueva = len(df_sin_exactos) - len(df_sin_exactos.drop_duplicates(subset=clave_nueva))
print(f"Residual clave NUEVA tras eliminar exactos: {dup_residual_nueva:,}")
print("(esperado 0 → no quedan duplicados semánticos)")

Residual clave NUEVA tras eliminar exactos: 0
(esperado 0 → no quedan duplicados semánticos)


In [ ]:
resumen = pd.DataFrame({
    "métrica": [
        "Duplicados exactos",
        "Duplicados clave VIEJA (sin periodo)",
        "Duplicados clave NUEVA (con periodo)",
        "Residual clave NUEVA tras eliminar exactos",
    ],
    "n_filas": [dup_exactos, dup_clave_vieja, dup_clave_nueva, dup_residual_nueva],
    "%_sobre_total": [
        round(dup_exactos / n * 100, 2),
        round(dup_clave_vieja / n * 100, 2),
        round(dup_clave_nueva / n * 100, 2),
        round(dup_residual_nueva / n * 100, 2),
    ],
})
print(resumen.to_string(index=False))

# La hipótesis se confirma si la clave NUEVA aísla exactamente los duplicados exactos
# (coincide con ellos) y no queda ningún duplicado semántico residual.
hipotesis_confirmada = (dup_clave_nueva == dup_exactos) and (dup_residual_nueva == 0)
print("\n¿Hipótesis confirmada (la clave NUEVA aísla exactamente los duplicados exactos y no deja residual)?:",
      "SÍ ✅" if hipotesis_confirmada else "NO ❌")
print("CLAVE_NUEVA a usar en process_matriculados.py:", clave_nueva)

                                   métrica  n_filas  %_sobre_total
                        Duplicados exactos   149539           4.34
      Duplicados clave VIEJA (sin periodo)  1493735          43.36
      Duplicados clave NUEVA (con periodo)   149539           4.34
Residual clave NUEVA tras eliminar exactos        0           0.00

¿Hipótesis confirmada (la clave NUEVA aísla exactamente los duplicados exactos y no deja residual)?: SÍ ✅
CLAVE_NUEVA a usar en process_matriculados.py: ['PERIODO_ESTANDARIZADO', 'CODIGO_INEI', 'CODIGO_SIU_PROGRAMA', 'CODIGO_GRUPO_1', 'CODIGO_GRUPO_3', 'CODIGO_LOCAL', 'GUID_PERSONA']


## Conclusión

El resultado **confirma la hipótesis**, con un matiz importante:

- Con la clave VIEJA (sin período) los duplicados se disparan a **43,36%** por dos razones: colisiones de la misma persona+programa entre semestres y el tratamiento `NaN == NaN` de pandas sobre `CODIGO_GRUPO_1/3`.
- Con la clave NUEVA (`[PERIODO_ESTANDARIZADO] + clave_vieja`) el conteo cae hasta **coincidir exactamente con los duplicados exactos** (misma fila repetida) y el **residual semántico es 0**.

> **`PERIODO_ESTANDARIZADO` debe formar parte de la clave de deduplicación.** La granularidad real de matriculados es *persona + programa + local + período*. Los únicos duplicados que quedan bajo esa clave son filas idénticas (basura del portal), que se eliminan con `drop_duplicates()`.

### Estrategia de deduplicación para el script de producción
1. `dropna` sobre `[PERIODO_ESTANDARIZADO] + clave_vieja` (nulos en columnas críticas).
2. `drop_duplicates(subset=CLAVE_NUEVA)` para eliminar los duplicados exactos (la clave NUEVA los aísla exactamente).

### Notas
- Este test usó **2 semestres** (2025-I y 2025-II). El conteo con clave VIEJA aquí (1.493.735) no es comparable con los 75.907 del contrato (que usó los 12 archivos y una heurística voraz distinta); lo relevante es la **diferencia** entre claves y el residual 0.
- La variable `CLAVE_NUEVA` queda definida para usarla literalmente en `src/process_matriculados.py`.
- La lógica de `Region_Sur` se agregará **después**, en el script de producción (igual que en Ingresantes), no en este experimento.

In [ ]:
# Después de cargar df (con los 2 archivos concatenados)
# Buscar las filas que están duplicadas exactamente
dup_mask = df.duplicated(keep=False)  # Marca todas las filas que tienen un duplicado
df_dup = df[dup_mask].sort_values(by=df.columns.tolist())  # Ordena para ver las duplicadas juntas

print(f"Filas duplicadas exactas: {len(df_dup):,}")
# Ver un par de ejemplos
df_dup.head(20)